# R20-H200 - Retrodictive test of expected-information-gain hypothesis selection

**Author**: KGF campaign executor  |  **Approach**: variational (active-inference-style) acquisition scored against the adjudicated ledger

Question (R20 preamble): can an expected-free-energy-style acquisition score - decomposed into an **epistemic** term (expected information gain about engine failure structure) and a **pragmatic** term (expected improvement of the engine objective) - retrodict which hypotheses in the 113-verdict history actually turned out informative?

**Acceptance bar (binding)**: Spearman rho >= 0.4 between EIG and realized informativeness AND AUC >= 0.7 separating the top-decile (promotions/flips) from the rest, on a **temporally honest** evaluation (each hypothesis scored using only verdicts recorded BEFORE its registration). Refuted below that -> R20 closes with H201 cancelled.

CPU-only, deterministic given the ledger. No graph, no GPU, no corpus content - this is a methods experiment about the campaign itself.

## Imports and configuration

In [1]:
import re, json, math
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

np.random.seed(0)

ROOT = Path('/home/lab/workspace/learning/projects/knowledge-graph-foundry')
LEDGER = ROOT / 'docs/experiments/kgf-redesign-experiments.md'
PROMOS = ROOT / 'docs/sota-promotions.md'
REPORTS = ROOT / 'reports'

FLANKS = ['identity','fidelity','retrieval','instrument','ops','structure']
print('ledger', LEDGER.exists(), 'promos', PROMOS.exists())

ledger True promos True


## 1. Parse the ledger into structured records

Each hypothesis lives under a `### R<round>-H<num> ...` header. Registration text = every `- **Field**` bullet before **Result** (Grounding/Weak-spot/Assumption/Hypothesis/Lever/Mechanism/Prediction/Acceptance bar/Experiment). Post-hoc = **Result** + **Verdict**. File order = registration order (rounds are dated; within-round order is file order). Canonical id = the global H-number (unique across the file, and the form used by downstream citations and the promotions ledger).

In [2]:
raw = LEDGER.read_text()
lines = raw.splitlines()

sec_starts = [i for i, ln in enumerate(lines) if ln.startswith('### ')]
records = []
for si, start in enumerate(sec_starts):
    end = sec_starts[si+1] if si+1 < len(sec_starts) else len(lines)
    header = lines[start][4:].strip()
    m = re.match(r'^(R(\d+))-H(\d+)([a-z]?)\b(.*)', header)
    if not m:
        continue  # non-hypothesis ### note (results/scoring)
    round_lbl, round_num, hnum, suf, title = m.group(1), int(m.group(2)), m.group(3), m.group(4), m.group(5).strip()
    hid = 'H' + hnum + suf
    body = lines[start:end]
    fields = {}
    cur = None
    for ln in body[1:]:
        fm = re.match(r'^- \*\*([^*]+)\*\*\s*-?\s*(.*)', ln)
        if fm:
            cur = fm.group(1).strip()
            fields.setdefault(cur, '')
            fields[cur] += fm.group(2) + '\n'
        elif cur is not None:
            fields[cur] += ln + '\n'
    records.append(dict(hid=hid, round=round_lbl, round_num=round_num, title=title,
                        reg_order=si, fields={k: v.strip() for k, v in fields.items()},
                        section_text='\n'.join(body)))

records.sort(key=lambda r: r['reg_order'])
for k, r in enumerate(records):
    r['reg_order'] = k
print('hypothesis sections parsed:', len(records))
print('rounds:', sorted(set(r['round'] for r in records), key=lambda x: int(x[1:])))

hypothesis sections parsed: 196
rounds: ['R01', 'R02', 'R03', 'R04', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20']


In [3]:
REG_FIELDS = ['Grounding','Weak spot','Assumption attacked','Prior art','Hypothesis',
              'Lever','Mechanism','Prediction','Acceptance bar','Experiment','Disqualifying',
              'Baseline graph','Engine','Data','Pipeline','Execution vehicle','Behind SOTA',
              'Temporal capability','Extraction recall proxy']

def reg_text(r):
    return '\n'.join(r['fields'].get(f,'') for f in REG_FIELDS if f in r['fields'])
def grounding_text(r):
    for f in ['Grounding','Weak spot','Assumption attacked']:
        if f in r['fields']: return r['fields'][f]
    return r['fields'].get('Hypothesis','')

for r in records:
    r['reg_text'] = reg_text(r)
    r['result'] = r['fields'].get('Result','')
    r['verdict'] = r['fields'].get('Verdict','')

dup = [h for h,c in Counter(r['hid'] for r in records).items() if c>1]
print('duplicate hids:', dup)

duplicate hids: []


## 2. Verdict classification (post-hoc)

Map the free-text verdict line to a class. Keyword priority: pending -> not adjudicated; REFUTED; CONFIRMED (incl. Promoted); PARTIAL; KEPT; INCONCLUSIVE. Separately flag **flip/contested** (the verdict superseded or contested a prior verdict, or flipped an earlier stance) via `contest|supersed|flip|overturn|vindicat` in the verdict text.

In [4]:
def classify_verdict(v):
    t = v.lower(); head = t[:40]
    if v.strip()=='' : return 'none'
    if 'pending' in head: return 'pending'
    if re.search(r'\brefuted\b', head) or head.startswith('not '): return 'REFUTED'
    if re.search(r'\b(confirmed|promoted)\b', head): return 'CONFIRMED'
    if re.search(r'partial', head): return 'PARTIAL'
    if re.search(r'\bkept\b', head): return 'KEPT'
    if re.search(r'inconclusive', head): return 'INCONCLUSIVE'
    if 'refuted' in t: return 'REFUTED'
    if 'confirmed' in t or 'promoted' in t: return 'CONFIRMED'
    if 'partial' in t: return 'PARTIAL'
    if 'kept' in t: return 'KEPT'
    if 'inconclusive' in t: return 'INCONCLUSIVE'
    return 'OTHER'

def is_flip(v):
    return bool(re.search(r'contest|supersed|flip|overturn|vindicat', v.lower()))

for r in records:
    r['vclass'] = classify_verdict(r['verdict'])
    r['flip'] = is_flip(r['verdict'])

print('verdict classes:', Counter(r['vclass'] for r in records))
adj = [r for r in records if r['vclass'] not in ('pending','none')]
print('adjudicated:', len(adj), '| flips among adjudicated:', sum(r['flip'] for r in adj))

verdict classes: Counter({'pending': 82, 'REFUTED': 56, 'CONFIRMED': 47, 'none': 5, 'KEPT': 3, 'INCONCLUSIVE': 2, 'PARTIAL': 1})
adjudicated: 109 | flips among adjudicated: 9


## 3. Pre-hoc features (registration text ONLY - never Result/Verdict)

- **flank** - keyword map into identity / fidelity / retrieval / instrument / ops / structure (argmax of keyword hits over registration text; round-theme fallback on ties)
- **contrarian vs conformist** - contrarian slates (R06-R08) and falsification language vs incremental deepening
- **cost class** - GPU (needs local model / completions) / ingest (rebuild / re-extraction) / CPU-deterministic (runs now / embeddings only / deterministic)
- **grounding cites anomaly vs confirmation** - does the grounding cite a prior failure/gap vs a prior confirmation
- **prediction hedge** - hedge-word density in the Prediction (lower = higher direction confidence)

In [5]:
FLANK_KW = {
 'identity': ['identit','resolver','resolution','merge','dedup','same_as','duplicate','alias','sibling','entity resolution','cross-type','canonical name','invariant','blocking'],
 'fidelity': ['extraction','extract','parse','parser','chunk','glyph','canonicaliz','pdf','ocr','segmentation','proposition split','ingest fidelity','name-recall','header carry','table row','variance'],
 'retrieval': ['retrieval','ppr','pagerank','hop','seed','recall','render','fanout','top_k','top-k','context','traversal','query','answerab','materializ','abstention','budget','multi-hop'],
 'instrument': ['benchmark','instrument','scorer','entailment','nli','calibrat','metric','probe','judge','measurement','comparator','fuzzy matcher','sufficiency','ece','labeled','ground truth','gold'],
 'ops': ['drift','longevity','mutation','scale','operational','provenance','revision','rebuild','curing','cure','temporal','bitemporal','supersed','shipping','generaliz','hardening','long-horizon'],
 'structure': ['topology','community','curvature','ontology','structural','graph shape','type consolidation','modularity','heaps','power-law','hub','communit'],
}
ROUND_THEME = {
 'R01':'retrieval','R02':'retrieval','R03':'retrieval','R04':'ops','R05':'ops','R06':'structure',
 'R07':'structure','R08':'structure','R09':'structure','R10':'structure','R11':'identity','R12':'identity',
 'R13':'identity','R14':'fidelity','R15':'ops','R16':'ops','R17':'ops','R18':'instrument','R19':'retrieval','R20':'instrument',
}

def flank_of(r):
    txt = (r['title']+'\n'+r['reg_text']).lower()
    scores = {f: sum(txt.count(k) for k in kws) for f,kws in FLANK_KW.items()}
    best = max(scores.values())
    if best==0: return ROUND_THEME.get(r['round'],'structure')
    winners = [f for f,s in scores.items() if s==best]
    if len(winners)==1: return winners[0]
    rt = ROUND_THEME.get(r['round'])
    return rt if rt in winners else sorted(winners)[0]

CONTRARIAN_KW = ['falsif','contrarian','demolition','refute its own','right to exist','load-bearing assumption',
                 'disprove','null hypothesis','challenge','never merge','must never']
def contrarian_of(r):
    if r['round'] in ('R06','R07','R08'): return 1
    if 'Assumption attacked' in r['fields']: return 1
    txt = r['reg_text'].lower()
    return 1 if sum(txt.count(k) for k in CONTRARIAN_KW) >= 2 else 0

def cost_of(r):
    txt = (r['fields'].get('Experiment','')+' '+r['reg_text']).lower()
    if any(k in txt for k in ['needs local model','local model','completions','gpu','llm judge','reader','adjudicat']):
        return 'GPU'
    if any(k in txt for k in ['rebuild','re-extraction','re-extract','ingest','wall-clock','wave','full engine','corpus rebuild']):
        return 'ingest'
    return 'CPU'

def grounds_anomaly(r):
    g = grounding_text(r).lower()
    anom = sum(g.count(k) for k in ['gap','fail','weak','defect','regression','unreliable','cannot','broken','anti-correlat','persistent','noise','starv','deficit','miss','loss','unrun','untested','vanity','wrong','stall'])
    conf = sum(g.count(k) for k in ['confirmed','promoted','validated','shipped','stands','held','decided','proven','robust'])
    return 1 if anom>=conf else 0

HEDGE = ['may ','likely','should','expect','either way',' if ','broadly','within','~','roughly','approximate','tend','around','some','partial','at least','>=','<=']
def pred_hedge(r):
    p = r['fields'].get('Prediction','').lower()
    if not p: return 0.5
    n = sum(p.count(k) for k in HEDGE); wc = max(len(p.split()),1)
    return min(n/(wc/20.0), 3.0)

for r in records:
    r['flank'] = flank_of(r); r['contrarian'] = contrarian_of(r)
    r['cost'] = cost_of(r); r['anomaly'] = grounds_anomaly(r); r['hedge'] = pred_hedge(r)

print('flank dist:', Counter(r['flank'] for r in records))
print('cost dist:', Counter(r['cost'] for r in records))
print('contrarian:', Counter(r['contrarian'] for r in records))
print('anomaly-cited:', Counter(r['anomaly'] for r in records))

flank dist: Counter({'retrieval': 51, 'instrument': 45, 'identity': 40, 'fidelity': 33, 'ops': 18, 'structure': 9})
cost dist: Counter({'GPU': 78, 'CPU': 73, 'ingest': 45})
contrarian: Counter({0: 162, 1: 34})
anomaly-cited: Counter({1: 184, 0: 12})


## 4. Post-hoc realized informativeness label

Four components, each in [0,1], combined by a stated weighting:

1. **verdict surprisal** - REFUTED = 1.0 (a refutation contradicts the hopeful prediction), INCONCLUSIVE = 0.7, PARTIAL/KEPT = 0.5, CONFIRMED = 0.0; **+0.5 if flip/contested** (capped at 1.0)
2. **promotion yield** - 1 if the id appears in `docs/sota-promotions.md`, else 0
3. **downstream registrations spawned** - count of *later* registrations whose registration text cites this id; `log1p`-normalized
4. **supersede/contest** - 1 if this verdict superseded or contested a prior verdict

Weights (simple, defensible): surprisal 0.35, promotion 0.30, downstream 0.25, supersede 0.10. Swept in section 7.

**Circularity caveat (honest)**: surprisal derives from the verdict class, and the surrogate belief model is *also* updated by verdict class - so EIG and the surprisal component share the verdict signal. Promotion and downstream are more independent. The surprisal-free sensitivity row isolates the shared channel.

In [6]:
promo_raw = PROMOS.read_text()
promo_ids = set(re.findall(r'\bH(\d+)\b', promo_raw))
print('distinct promoted H-ids:', len(promo_ids))

def cites(text, hid):
    return re.search(r'\b'+re.escape(hid)+r'\b', text) is not None
for r in records:
    later = [q for q in records if q['reg_order'] > r['reg_order']]
    r['downstream'] = sum(cites(q['reg_text'], r['hid']) for q in later)

SURP = {'REFUTED':1.0,'INCONCLUSIVE':0.7,'PARTIAL':0.5,'KEPT':0.5,'CONFIRMED':0.0,'OTHER':0.4}
max_down = max((r['downstream'] for r in records), default=1)
log_max = math.log1p(max_down) or 1.0
for r in records:
    r['c_surprisal'] = min(SURP.get(r['vclass'],0.4) + (0.5 if r['flip'] else 0.0), 1.0)
    r['c_promo'] = 1.0 if re.sub(r'\D','',r['hid']) in promo_ids else 0.0
    r['c_down'] = math.log1p(r['downstream'])/log_max
    r['c_supersede'] = 1.0 if r['flip'] else 0.0

W = dict(surprisal=0.35, promo=0.30, down=0.25, supersede=0.10)
def informativeness(r, w=W):
    return (w['surprisal']*r['c_surprisal'] + w['promo']*r['c_promo']
            + w['down']*r['c_down'] + w['supersede']*r['c_supersede'])
for r in records:
    r['realized'] = informativeness(r)

adj = [r for r in records if r['vclass'] not in ('pending','none','OTHER')]
adj.sort(key=lambda r: r['reg_order'])
print('adjudicated for scoring:', len(adj))
print('promoted among adjudicated:', int(sum(r['c_promo'] for r in adj)))
print('mean realized:', round(np.mean([r['realized'] for r in adj]),3))

distinct promoted H-ids: 51


adjudicated for scoring: 109
promoted among adjudicated: 46
mean realized: 0.406


## 5. Surrogate belief model + EIG (temporally honest)

Latent per-flank defect rate theta_s ~ Beta(alpha_s, beta_s), initialized Beta(1,1). Outcome mapping per hypothesis in flank s: `y=1` (defect evidence) if the verdict is REFUTED / PARTIAL / INCONCLUSIVE / flipped (the probed subsystem remained/was-shown defective, or an engine assumption fell); `y=0` if CONFIRMED/KEPT (the lever held / mechanism worked). Updated **sequentially in registration order**; only adjudicated hypotheses move the state.

Per-hypothesis **EIG** = mutual information I(theta_s ; Y) between the (as-yet-unobserved) Bernoulli outcome and the latent, computed from the flank's Beta state **before** this hypothesis's registration - so it uses only earlier verdicts. I(theta;Y) = H(Y) - E_theta[H(Y|theta)], on a fine theta grid.

In [7]:
from scipy.stats import beta as Beta
def defect_outcome(r):
    return 1 if (r['vclass'] in ('REFUTED','PARTIAL','INCONCLUSIVE') or r['flip']) else 0

GRID = np.linspace(1e-4, 1-1e-4, 400)
def bin_ent(p):
    p = np.clip(p, 1e-12, 1-1e-12)
    return -(p*np.log2(p) + (1-p)*np.log2(1-p))
def eig_mi(a,b):
    pdf = Beta.pdf(GRID,a,b); pdf = pdf/np.trapezoid(pdf,GRID)
    p_marg = np.trapezoid(GRID*pdf, GRID)
    HY = bin_ent(p_marg)
    EH = np.trapezoid(bin_ent(GRID)*pdf, GRID)
    return float(max(HY - EH, 0.0))

state = {f: [1.0,1.0] for f in FLANKS}
for r in records:
    a,b = state[r['flank']]
    r['eig'] = eig_mi(a,b)
    r['flank_alpha_pre'], r['flank_beta_pre'] = a, b
    if r['vclass'] not in ('pending','none','OTHER'):
        y = defect_outcome(r)
        if y==1: state[r['flank']][0]+=1
        else:    state[r['flank']][1]+=1

print('final per-flank posterior mean defect rate:')
for f in FLANKS:
    a,b=state[f]; print(f'  {f:10s} Beta({a:.0f},{b:.0f}) mean={a/(a+b):.2f}')
print('EIG range on adjudicated:', round(min(r['eig'] for r in adj),4),'..',round(max(r['eig'] for r in adj),4))

final per-flank posterior mean defect rate:
  identity   Beta(24,8) mean=0.75
  fidelity   Beta(6,11) mean=0.35
  retrieval  Beta(18,16) mean=0.53
  instrument Beta(14,8) mean=0.64
  ops        Beta(5,4) mean=0.56
  structure  Beta(2,5) mean=0.29
EIG range on adjudicated: 0.0215 .. 0.2785


## 6. Pragmatic term and expected-free-energy variants

The **pragmatic** value proxies expected improvement of the engine objective (Phi) from the registration's cost/lever class: a cheap, direct lever scores high. `pragmatic = lever_bonus * cheapness`, cheapness = {CPU 1.0, ingest 0.6, GPU 0.4}, lever_bonus = 1.0 if the section carries a Lever/Mechanism field or fix language else 0.5. **EFE** = epistemic (EIG) + pragmatic (both min-max normalized).

In [8]:
CHEAP = {'CPU':1.0,'ingest':0.6,'GPU':0.4}
def pragmatic_of(r):
    lever = 1.0 if ('Lever' in r['fields'] or 'Mechanism' in r['fields']
                    or re.search(r'ships|lever|fix|operator|cut|lift|improv|reduce', r['reg_text'].lower())) else 0.5
    return lever*CHEAP[r['cost']]
for r in records:
    r['pragmatic'] = pragmatic_of(r)

def mm(xs):
    xs=np.asarray(xs,float); lo,hi=xs.min(),xs.max()
    return (xs-lo)/(hi-lo+1e-12)
A = adj
eig_n = mm([r['eig'] for r in A]); prag_n = mm([r['pragmatic'] for r in A])
for r,e,p in zip(A,eig_n,prag_n):
    r['eig_n'], r['prag_n'] = float(e), float(p); r['efe'] = float(e+p)
print('pragmatic mean by cost:', {c: round(np.mean([r['pragmatic'] for r in A if r['cost']==c]),2) for c in CHEAP})

pragmatic mean by cost: {'CPU': np.float64(0.74), 'ingest': np.float64(0.47), 'GPU': np.float64(0.33)}


## 7. Scoring: rho, AUC, epistemic-vs-pragmatic, sensitivity

In [9]:
realized = np.array([r['realized'] for r in A])
eig  = np.array([r['eig'] for r in A])
prag = np.array([r['pragmatic'] for r in A])
efe  = np.array([r['efe'] for r in A])

rho_eig  = spearmanr(eig, realized).correlation
rho_prag = spearmanr(prag, realized).correlation
rho_efe  = spearmanr(efe, realized).correlation

pos = np.array([1 if (r['c_promo']>0 or r['flip']) else 0 for r in A])
def safe_auc(y,s):
    return float(roc_auc_score(y,s)) if len(set(y))>1 else float('nan')
auc_eig = safe_auc(pos, eig); auc_prag = safe_auc(pos, prag); auc_efe = safe_auc(pos, efe)

thr = np.quantile(realized, 0.9)
pos_dec = (realized>=thr).astype(int)
auc_eig_dec = safe_auc(pos_dec, eig)

print(f'n adjudicated = {len(A)} | positive (promo|flip) = {int(pos.sum())}')
print(f'rho  EIG={rho_eig:.3f}  pragmatic={rho_prag:.3f}  EFE={rho_efe:.3f}   (bar >= 0.40)')
print(f'AUC(promo|flip)  EIG={auc_eig:.3f}  pragmatic={auc_prag:.3f}  EFE={auc_efe:.3f}   (bar >= 0.70)')
print(f'AUC(top-decile realized)  EIG={auc_eig_dec:.3f}')
print(f'epistemic-vs-pragmatic: EIG retrodicts {"BETTER" if rho_eig>rho_prag else "WORSE"} than pragmatic')

n adjudicated = 109 | positive (promo|flip) = 52
rho  EIG=-0.185  pragmatic=-0.165  EFE=-0.290   (bar >= 0.40)
AUC(promo|flip)  EIG=0.426  pragmatic=0.488  EFE=0.432   (bar >= 0.70)
AUC(top-decile realized)  EIG=0.351
epistemic-vs-pragmatic: EIG retrodicts WORSE than pragmatic


In [10]:
schemes = {
 'baseline (0.35/0.30/0.25/0.10)': dict(surprisal=.35,promo=.30,down=.25,supersede=.10),
 'equal':                          dict(surprisal=.25,promo=.25,down=.25,supersede=.25),
 'promo-heavy':                    dict(surprisal=.15,promo=.50,down=.25,supersede=.10),
 'downstream-heavy':               dict(surprisal=.15,promo=.25,down=.50,supersede=.10),
 'surprisal-free (circularity)':   dict(surprisal=.00,promo=.45,down=.40,supersede=.15),
}
sens=[]
for name,w in schemes.items():
    rz=np.array([informativeness(r,w) for r in A])
    rr=spearmanr(eig,rz).correlation
    thr2=np.quantile(rz,0.9); yd=(rz>=thr2).astype(int)
    sens.append((name, round(rr,3), round(safe_auc(pos,eig),3), round(safe_auc(yd,eig),3)))
print('scheme | rho(EIG,realized) | AUC(promo|flip) | AUC(top-decile)')
for s in sens: print(f'  {s[0]:34s} rho={s[1]:+.3f}  AUC_pf={s[2]:.3f}  AUC_dec={s[3]:.3f}')

scheme | rho(EIG,realized) | AUC(promo|flip) | AUC(top-decile)
  baseline (0.35/0.30/0.25/0.10)     rho=-0.185  AUC_pf=0.426  AUC_dec=0.351
  equal                              rho=-0.195  AUC_pf=0.426  AUC_dec=0.322
  promo-heavy                        rho=-0.207  AUC_pf=0.426  AUC_dec=0.314
  downstream-heavy                   rho=-0.181  AUC_pf=0.426  AUC_dec=0.417
  surprisal-free (circularity)       rho=-0.119  AUC_pf=0.426  AUC_dec=0.444


## 8. Top-10 EIG picks vs top-10 realized informative - the interpretable payload

In [11]:
def top10(key):
    return sorted(A, key=lambda r:(-r[key], r['reg_order']))[:10]
top_eig = top10('eig'); top_real = top10('realized')
set_eig = {r['hid'] for r in top_eig}; set_real={r['hid'] for r in top_real}
overlap = set_eig & set_real
print('TOP-10 by EIG (model pick):')
for r in top_eig:
    print(f"  {r['hid']:5s} {r['round']} flank={r['flank']:10s} eig={r['eig']:.3f} realized={r['realized']:.2f} v={r['vclass'][:4]} promo={int(r['c_promo'])} flip={int(r['flip'])}")
print('\nTOP-10 by realized informativeness (ground truth):')
for r in top_real:
    print(f"  {r['hid']:5s} {r['round']} flank={r['flank']:10s} eig={r['eig']:.3f} realized={r['realized']:.2f} v={r['vclass'][:4]} promo={int(r['c_promo'])} flip={int(r['flip'])}")
print(f'\nOVERLAP {len(overlap)}/10:', sorted(overlap, key=lambda h:int(re.sub(r"\D","",h))))

named = ['H107','H51','H172','H194','H184']
print('\nregistered-prediction picks EIG percentile among adjudicated:')
for h in named:
    hit=[r for r in A if r['hid']==h]
    if hit:
        r=hit[0]; pct=100*sum(1 for q in A if q['eig']<=r['eig'])/len(A)
        print(f"  {h}: eig={r['eig']:.3f} pct={pct:.0f} realized={r['realized']:.2f} flank={r['flank']} v={r['vclass']}")
    else:
        print(f"  {h}: not in adjudicated set")

TOP-10 by EIG (model pick):
  H1    R01 flank=ops        eig=0.279 realized=0.06 v=CONF promo=0 flip=0
  H2    R01 flank=retrieval  eig=0.279 realized=0.06 v=CONF promo=0 flip=0
  H3    R01 flank=fidelity   eig=0.279 realized=0.06 v=CONF promo=0 flip=0
  H4    R01 flank=identity   eig=0.279 realized=0.00 v=CONF promo=0 flip=0
  H22   R04 flank=instrument eig=0.279 realized=0.21 v=CONF promo=0 flip=0
  H68   R09 flank=structure  eig=0.279 realized=0.11 v=CONF promo=0 flip=0
  H5    R01 flank=identity   eig=0.197 realized=0.45 v=KEPT promo=0 flip=1
  H6    R01 flank=retrieval  eig=0.197 realized=0.06 v=CONF promo=0 flip=0
  H7    R01 flank=ops        eig=0.197 realized=0.00 v=CONF promo=0 flip=0
  H10   R02 flank=fidelity   eig=0.197 realized=0.23 v=KEPT promo=0 flip=0

TOP-10 by realized informativeness (ground truth):
  H101  R11 flank=instrument eig=0.075 realized=0.84 v=REFU promo=1 flip=0
  H193  R19 flank=fidelity   eig=0.043 realized=0.81 v=REFU promo=1 flip=1
  H54   R08 flank=id

## 9. Verdict against the registered bars + write report

In [12]:
rho_pass = rho_eig >= 0.40
auc_pass = (auc_eig >= 0.70) or (auc_eig_dec >= 0.70)
verdict = 'CONFIRMED' if (rho_pass and auc_pass) else 'REFUTED'
epistemic_wins = rho_eig > rho_prag

ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report = dict(
  experiment='R20-H200', timestamp=ts, n_adjudicated=len(A),
  bars=dict(rho_min=0.40, auc_min=0.70),
  results=dict(rho_eig=rho_eig, rho_pragmatic=rho_prag, rho_efe=rho_efe,
               auc_eig_promoflip=auc_eig, auc_pragmatic=auc_prag, auc_efe=auc_efe,
               auc_eig_topdecile=auc_eig_dec),
  rho_pass=bool(rho_pass), auc_pass=bool(auc_pass),
  epistemic_beats_pragmatic=bool(epistemic_wins),
  verdict=verdict, informativeness_weights=W,
  sensitivity=[dict(scheme=s[0], rho=s[1], auc_promoflip=s[2], auc_topdecile=s[3]) for s in sens],
  top10_eig=[dict(hid=r['hid'],round=r['round'],flank=r['flank'],eig=round(r['eig'],4),
                  realized=round(r['realized'],3),vclass=r['vclass'],promo=bool(r['c_promo']),flip=bool(r['flip'])) for r in top_eig],
  top10_realized=[dict(hid=r['hid'],round=r['round'],flank=r['flank'],eig=round(r['eig'],4),
                  realized=round(r['realized'],3),vclass=r['vclass'],promo=bool(r['c_promo']),flip=bool(r['flip'])) for r in top_real],
  top10_overlap=sorted(overlap),
  final_flank_defect_rates={f:(state[f][0]/(state[f][0]+state[f][1])) for f in FLANKS},
)
out = REPORTS / f'variational-h200-{ts}.json'
out.write_text(json.dumps(report, indent=2))
print('VERDICT:', verdict)
print(f'rho_eig={rho_eig:.3f} (bar 0.40, {"PASS" if rho_pass else "FAIL"}) | '
      f'auc_eig={auc_eig:.3f}/topdec={auc_eig_dec:.3f} (bar 0.70, {"PASS" if auc_pass else "FAIL"})')
print('epistemic beats pragmatic:', epistemic_wins)
print('wrote', out)

VERDICT: REFUTED
rho_eig=-0.185 (bar 0.40, FAIL) | auc_eig=0.426/topdec=0.351 (bar 0.70, FAIL)
epistemic beats pragmatic: False
wrote /home/lab/workspace/learning/projects/knowledge-graph-foundry/reports/variational-h200-20260707T151833Z.json
